# Exemplo: Deep learning
------------------------

Este exemplo mostra como usar o ExperionML para treinar e validar uma rede neural convolucional implementada com [Keras](https://keras.io/) usando [scikeras](https://www.adriangb.com/scikeras/refs/heads/master/index.html).

Importe o conjunto de dados MNIST de [keras.datasets](https://keras.io/api/datasets/mnist/). Este é um conhecido conjunto de imagens cujo objetivo é classificar dígitos manuscritos.

## Carregar os dados

In [ ]:
!pip install tensorflow>=2.16.2

In [ ]:
# Disable annoying tf warnings
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from tensorflow import get_logger
get_logger().setLevel("ERROR")

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

from experionml import ExperionMLClassifier, ExperionMLModel
from sklearn.preprocessing import FunctionTransformer
from optuna.pruners import PatientPruner
from optuna.distributions import CategoricalDistribution, IntDistribution

from scikeras.wrappers import KerasClassifier
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Dropout

In [ ]:
# Baixe o conjunto de dados MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Flatten data to follow sklearn's API (2d input)
X_train = X_train.reshape(len(X_train), -1)
X_test = X_test.reshape(len(X_test), -1)

data = (X_train, y_train), (X_test, y_test)

In [ ]:
# Crie a rede neural convolucional
class ConvNN(KerasClassifier):
    """Convolutional neural network model."""

    def __repr__(self):
        return f"ConvCNN(epochs={self.epochs}, batch_size={self.batch_size})"
    
    @property
    def feature_encoder(self):
        """Convert the 2d input to the image's format (len(X), 28, 28, 1)."""
        return FunctionTransformer(
            func=lambda X: X.reshape(X.shape[0], 28, 28, 1),
        )

    @staticmethod
    def _keras_build_fn(**kwargs):
        """Create the model's architecture."""
        model = Sequential()
        model.add(
            Conv2D(
                filters=8,
                kernel_size=3,
                activation="relu",
                input_shape=(28, 28, 1),
            )
        )
        model.add(Conv2D(filters=4, kernel_size=5, activation="relu"))
        model.add(Flatten())
        model.add(Dense(units=10, activation="softmax"))
        model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
        )

        return model

In [ ]:
# Converta o modelo em um modelo do ExperionML
model = ExperionMLModel(
    estimator=ConvNN(verbose=0),
    acronym="CNN",
    needs_scaling=True,  # Applies automated feature scaling before fitting
    validation="epochs",  # Aplica validação durante o treino ao parâmetro epochs
)

## Executar o pipeline

In [ ]:
experionml = ExperionMLClassifier(*data, n_rows=0.1, verbose=2, random_state=1)

In [ ]:
# Como em qualquer outro modelo, podemos definir distribuições personalizadas para ajuste de hiperparâmetros
experionml.run(
    models=model,
    metric="f1_weighted",
    n_trials=12,
    ht_params={
        "distributions": {
            "epochs": IntDistribution(2, 10),
            "batch_size": CategoricalDistribution([128, 256, 512]),
        },
    },
    errors='raise'
)

## Analisar os resultados

In [ ]:
experionml.cnn.trials

In [ ]:
experionml.plot_evals(dataset="test+train")

In [ ]:
# Use os métodos de predição como em qualquer outro modelo
experionml.cnn.predict_proba(X_train)

In [ ]:
# Ou gerar gráficos...
experionml.cnn.plot_hyperparameters()

In [ ]:
experionml.plot_parallel_coordinate()